In [ ]:
import sys
sys.path.append("/project/src")

In [ ]:
from coxmodel import CoxPH
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceWarning
from lifelines import KaplanMeierFitter
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, validation_curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
import joblib
import warnings
from scipy.stats import randint, uniform, loguniform
from scipy.interpolate import interp1d
from sklearn.utils.class_weight import compute_sample_weight
from sksurv.metrics import integrated_brier_score, cumulative_dynamic_auc
from sksurv.util import Surv
from sklearn.metrics import mean_squared_error

from preprocessing import (
    split_features_target,
    build_survival_target,
    low_missingness_complete_case_analysis,
    concat_features_target,
    decode_preprocessed_feature_name,
    SURVIVAL_EVENT_COL,
    SURVIVAL_TIME_COL,
)

In [ ]:
def cox_concordance_scorer(estimator, X, y):
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        try:
            return estimator.score(X, y)
        except ConvergenceWarning:
            return np.nan

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    train_df_csv = "/content/drive/MyDrive/bachelor/train_preprocessed.csv"
    test_df_csv = "/content/drive/MyDrive/bachelor/test_preprocessed.csv"
else:
    train_df_csv = "./data/train_preprocessed.csv"
    test_df_csv = "./data/test_preprocessed.csv"

In [ ]:
train_df = pd.read_csv(train_df_csv, delimiter=',')
test_df = pd.read_csv(test_df_csv, delimiter=',')

In [ ]:
train_X, train_y = split_features_target(train_df)
test_X, test_y = split_features_target(test_df)

In [ ]:
best_parameters = joblib.load("joblib-storage/cox_best_params.joblib")

cox = CoxPH(
    penalizer=best_parameters["penalizer"],
    l1_ratio=best_parameters["l1_ratio"],
    duration_col=SURVIVAL_TIME_COL,
    event_col=SURVIVAL_EVENT_COL,
    compute_weights=True,
)

cv        = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_splits = list(cv.split(train_X, train_y[SURVIVAL_EVENT_COL].astype(int)))

run = wandb.init(
    entity="semariik",
    project="survival-analysis-mci",
)

In [ ]:
cox.fit(train_X, train_y)

In [ ]:
cross_val_scores = cross_val_score(
    cox, train_X, train_y,
    cv=cv_splits,
    scoring=cox_concordance_scorer,
)
print(f"CV C-index: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

In [ ]:
cindex_test = cox.score(test_X, test_y)
print(f"Test C-index: {cindex_test:.4f}")

In [ ]:
# Time grid: from earliest observed event to 90th percentile of times
t_min     = float(test_y[SURVIVAL_TIME_COL][test_y[SURVIVAL_EVENT_COL]].min())
t_max     = float(np.percentile(test_y[SURVIVAL_TIME_COL], 90))
times_ibs = np.linspace(t_min, t_max, 100)

# Survival matrix: shape (n_subjects, n_times)
surv_funcs  = cox.predict_survival_function(test_X)
surv_matrix = np.zeros((len(test_X), len(times_ibs)))
for i, col in enumerate(surv_funcs.columns):
    f = interp1d(
        surv_funcs.index, surv_funcs[col],
        kind='previous', bounds_error=False,
        fill_value=(1.0, float(surv_funcs[col].iloc[-1])),
    )
    surv_matrix[i] = f(times_ibs)

ibs = integrated_brier_score(train_y, test_y, surv_matrix, times_ibs)
print(f"Integrated Brier Score: {ibs:.4f}")

wandb.log({"eval/integrated_brier_score": ibs})

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(test_y[SURVIVAL_TIME_COL], event_observed=test_y[SURVIVAL_EVENT_COL].astype(bool))

mean_surv = cox.predict_survival_function(test_X).mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
kmf.plot_survival_function(ax=ax, label="Kaplan-Meier (observed)", color="steelblue")
ax.plot(mean_surv.index, mean_surv.values, color="tomato", linewidth=2, label="Cox model (mean predicted)")
ax.set_xlabel("Time (years)", fontsize=12)
ax.set_ylabel("Survival probability", fontsize=12)
ax.set_title("Kaplan-Meier vs Cox Model — Survival Curve Comparison", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("cox_km_vs_model.png", dpi=150, bbox_inches="tight")
plt.show()

wandb.log({"eval/km_vs_cox": wandb.Image("cox_km_vs_model.png")})

In [ ]:
times_roc   = [1.0, 2.0, 3.0]
risk_scores = cox.predict_partial_hazard(test_X).values

auc_scores, mean_auc = cumulative_dynamic_auc(train_y, test_y, risk_scores, times_roc)

for t, a in zip(times_roc, auc_scores):
    print(f"AUC at {t:.0f} year(s): {a:.4f}")
print(f"Mean AUC:          {mean_auc:.4f}")

wandb.log({
    "eval/auc_1yr":  auc_scores[0],
    "eval/auc_2yr":  auc_scores[1],
    "eval/auc_3yr":  auc_scores[2],
    "eval/mean_auc": mean_auc,
})